# Tuning pipeline: vocab variants, Bag-NN and SVM search

此 notebook 会：
- 为若干 (vocab_size, dsift_step) 组合构建并保存 `vocab_<size>_s<step>.pkl`。
- 用每个 vocab 生成 Bag-of-SIFT 特征（可指定 `step`）。
- 对 Bag-NN 做 k 搜索并记录最优 k 与准确率。
- 对 SVM 在 `raw` 与 `sqrt_l2` 特征变换上做小规模网格搜索（C, loss, class_weight）。
- 保存每次试验结果与混淆矩阵，最后汇总最优方案。

In [1]:
import os, os.path as osp, pickle
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.svm import LinearSVC
from utils import get_image_paths
import student_code_12311805 as sc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT = 'outputs/tuning'
os.makedirs(OUT, exist_ok=True)
categories = ['Kitchen','Store','Bedroom','LivingRoom','Office','Industrial','Suburb','InsideCity','TallBuilding','Street','Highway','OpenCountry','Coast','Mountain','Forest']
data_path = osp.join('..','data')
train_paths, test_paths, train_labels, test_labels = get_image_paths(data_path, categories, 100)
print('Loaded', len(train_paths), 'train images and', len(test_paths),'test images')

Loaded 1500 train images and 1500 test images


In [2]:
# vocab and step choices (small grid to limit runtime)
vocab_sizes = [200, 400]
steps = [15, 10]
results = []

for V in vocab_sizes:
    for S in steps:
        name = f'vocab_{V}_s{S}.pkl'
        path = osp.join(OUT, name)
        if not osp.isfile(path):
            print('Building', name)
            vocab = sc.build_vocabulary(train_paths, V, step=S, sample_per_image=100)
            with open(path,'wb') as f:
                pickle.dump(vocab,f)
        else:
            print('Loading', name)
            with open(path,'rb') as f:
                vocab = pickle.load(f)

        # compute bags (use same step for quantization)
        print('Computing bags for', name)
        train_bags = sc.get_bags_of_sifts(train_paths, path, step=S)
        test_bags = sc.get_bags_of_sifts(test_paths, path, step=S)

        # NN k-search (chi2)
        from sklearn.metrics import pairwise_distances
        def chi2_dist(A,B):
            eps=1e-10
            D=np.zeros((A.shape[0],B.shape[0]),dtype=np.float32)
            for i,a in enumerate(A):
                D[i]=0.5*np.sum((B-a)**2/(B+a+eps),axis=1)
            return D
        D=chi2_dist(np.asarray(test_bags), np.asarray(train_bags))
        best_k_res=(None,-1)
        for k in [1,3,5,7,9]:
            idx=np.argpartition(D,kth=k-1,axis=1)[:,:k]
            preds=[]
            train_labels_np=np.asarray(train_labels)
            for i,ns in enumerate(idx):
                labs=train_labels_np[ns]
                if k==1:
                    preds.append(labs[0]); continue
                # majority vote
                vals,counts=np.unique(labs,return_counts=True)
                maxc=counts.max(); cand=vals[counts==maxc]
                if len(cand)==1:
                    preds.append(cand[0]); continue
                # tie break by sum of inverse distances
                best=None; bw=-1
                for c in cand:
                    mask=(labs==c)
                    w=np.sum(1.0/(D[i,ns[mask]]+1e-10))
                    if w>bw: bw=w; best=c
                preds.append(best)
            cm=confusion_matrix([categories.index(l) for l in test_labels],[categories.index(p) for p in preds], labels=list(range(len(categories))))
            acc=np.mean(np.diag(cm)/cm.sum(axis=1))
            if acc>best_k_res[1]: best_k_res=(k,acc)
        print('Best k for',name,':',best_k_res)
        # SVM small grid: feature modes and C values
        cfg_best=(None, -1, None)
        for mode in ['raw','sqrt_l2']:
            if mode=='raw':
                Xtr=np.asarray(train_bags); Xte=np.asarray(test_bags)
            else:
                Xtr=np.sqrt(np.clip(np.asarray(train_bags),0,None)); Xte=np.sqrt(np.clip(np.asarray(test_bags),0,None))
                Xtr=normalize(Xtr,norm='l2'); Xte=normalize(Xte,norm='l2')
            if mode=='raw':
                scaler=StandardScaler(); Xtr_s=scaler.fit_transform(Xtr); Xte_s=scaler.transform(Xte)
            else:
                Xtr_s=Xtr; Xte_s=Xte
            for C in [0.01,0.1,1.0,3.0]:
                for loss in ['hinge','squared_hinge']:
                    for cw in [None,'balanced']:
                        svms={}
                        for cat in categories:
                            y=np.array([1 if l==cat else -1 for l in train_labels])
                            clf=LinearSVC(C=C,loss=loss,class_weight=cw,dual=True,max_iter=50000,random_state=0,tol=1e-4)
                            clf.fit(Xtr_s,y)
                            svms[cat]=clf
                        S=np.vstack([svms[c].decision_function(Xte_s) for c in categories]).T
                        pred=[categories[i] for i in np.argmax(S,axis=1)]
                        cm=confusion_matrix([categories.index(l) for l in test_labels],[categories.index(p) for p in pred], labels=list(range(len(categories))))
                        acc=np.mean(np.diag(cm)/cm.sum(axis=1))
                        if acc>cfg_best[1]: cfg_best=((mode,C,loss,cw),acc,cm)
        print('Best SVM for',name,':',cfg_best[0],cfg_best[1])
        results.append({'vocab':path,'vocab_size':V,'step':S,'best_k':best_k_res,'best_svm':cfg_best})
# save summary
with open(osp.join(OUT,'summary.pkl'),'wb') as f:
    pickle.dump(results,f)

print('Done. Results saved to',OUT)

Building vocab_200_s15.pkl
Computing bags for vocab_200_s15.pkl
Best k for vocab_200_s15.pkl : (7, 0.5513333333333335)


/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  

Best SVM for vocab_200_s15.pkl : ('sqrt_l2', 0.1, 'squared_hinge', 'balanced') 0.6326666666666667
Building vocab_200_s10.pkl
Computing bags for vocab_200_s10.pkl
Best k for vocab_200_s10.pkl : (9, 0.5786666666666667)


/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/data2/cse12311805/miniconda/envs/assignment3/lib/python3.8/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best SVM for vocab_200_s10.pkl : ('sqrt_l2', 1.0, 'squared_hinge', None) 0.6853333333333331
Building vocab_400_s15.pkl
Computing bags for vocab_400_s15.pkl
Best k for vocab_400_s15.pkl : (7, 0.5459999999999999)
Best SVM for vocab_400_s15.pkl : ('sqrt_l2', 0.1, 'squared_hinge', 'balanced') 0.6413333333333333
Building vocab_400_s10.pkl
Computing bags for vocab_400_s10.pkl
Best k for vocab_400_s10.pkl : (7, 0.592)
Best SVM for vocab_400_s10.pkl : ('sqrt_l2', 1.0, 'squared_hinge', None) 0.6999999999999998
Done. Results saved to outputs/tuning
